# MODULES

In [1]:
!pip install pathlib
!pip install matplotlib

In [2]:
!nvidia-smi

Wed Sep 17 13:33:30 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce MX330           Off |   00000000:2C:00.0 Off |                  N/A |
| N/A   49C    P8            N/A  / 5001W |       5MiB /   2048MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

1.13.1+cu117
11.7
NVIDIA GeForce MX330


In [4]:
import cv2
import os
import math
import time
import random
import pathlib
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt

# GLOBAL VARIABLES

In [5]:
ROOT_DIR_IMAGES = '../Kaggle'

# IMAGES

In [26]:
def get_image_paths(root_dir, include='*', exclude=[]):
  paths = []
  
  for path in pathlib.Path(root_dir).glob(include): 
    paths.append(path)

  paths = sorted(paths)

  return paths

    
def base_filename_organization(imagPaths):
    baseFileNames = {}
    for imagPath in imagPaths:
        baseFileName = '_'.join(str(imagPath.stem).split('_')[:-1])
        if baseFileName not in baseFileNames:
            baseFileNames[baseFileName] = []
        baseFileNames[baseFileName].append(imagPath)

    # Sort lists numerically by the last suffix
    for key in baseFileNames:
        baseFileNames[key].sort(
            key=lambda p: int(p.stem.split('_')[-1])
        )

    return baseFileNames


def remove_last_element(baseFileNames):
    for baseFileName in baseFileNames:
        imagPaths = baseFileNames[baseFileName]
        length = len(imagPaths)
        baseFileNameLength = baseFileName + f'_{length-1}'
        imagPaths = [imagPath for imagPath in imagPaths if baseFileNameLength not in str(imagPath)]
        baseFileNames[baseFileName] = imagPaths
    return baseFileNames


def make_composite_images(images_dict: dict, final_width: int, final_height: int, save_dir: str):
    """
    Creates composite images from dict of images using OpenCV.

    Args:
        images_dict (dict): {key: [list of image paths]}
        final_width (int): width of final composite image
        final_height (int): height of final composite image
        save_dir (str): directory to save composite results
    """
    pathlib.Path(save_dir).mkdir(parents=True, exist_ok=True)

    for key, img_paths in images_dict.items():
        n = len(img_paths)
        grid_size = math.ceil(math.sqrt(n))  # square-ish grid
        cell_w = final_width // grid_size
        cell_h = final_height // grid_size

        # Black canvas
        composite = np.zeros((final_height, final_width, 3), dtype=np.uint8)

        for idx, img_path in enumerate(img_paths):
            try:
                img = cv2.imread(str(img_path))
                if img is None:
                    print(f"⚠️ Could not load {img_path}")
                    continue

                img = cv2.resize(img, (cell_w, cell_h), interpolation=cv2.INTER_AREA)

                row, col = divmod(idx, grid_size)
                y, x = row * cell_h, col * cell_w
                composite[y:y+cell_h, x:x+cell_w] = img
            except Exception as e:
                print(f"⚠️ Error with {img_path}: {e}")

        out_path = str(pathlib.Path(save_dir) / f"{key}.jpg")
        cv2.imwrite(out_path, composite)
        print(f"✅ Saved {out_path}")
        return

In [27]:
imagsPaths = {}
imagsPaths['fall']    = get_image_paths( ROOT_DIR_IMAGES + '/Fall/Raw_Image')
imagsPaths['no_fall'] = get_image_paths( ROOT_DIR_IMAGES + '/No_Fall/Raw_Image')

print(f"[INFO] videos dict keys: {list(imagsPaths.keys())}")
print(f"[INFO] videos['fall'] has {len(imagsPaths['fall'])} entries")
print(f"[INFO] videos['no_fall'] has {len(imagsPaths['no_fall'])} entries")

[INFO] videos dict keys: ['fall', 'no_fall']
[INFO] videos['fall'] has 49985 entries
[INFO] videos['no_fall'] has 61029 entries


In [28]:
BASEFILENAMES = {}

BASEFILENAMES['fall'] = base_filename_organization(imagsPaths['fall'])
BASEFILENAMES['no_fall'] = base_filename_organization(imagsPaths['no_fall'])
print( f"Fall {len(BASEFILENAMES['fall'])}")
print( f"No fall {len(BASEFILENAMES['no_fall'])}")


Fall 3136
No fall 3822


## Remove last image

In [29]:
BASEFILENAMES['fall'] = remove_last_element(BASEFILENAMES['fall'])
BASEFILENAMES['no_fall'] = remove_last_element(BASEFILENAMES['no_fall'])
print( f"Fall {len(BASEFILENAMES['fall'])}")
print( f"No fall {len(BASEFILENAMES['no_fall'])}")


Fall 3136
No fall 3822


In [30]:
make_composite_images(BASEFILENAMES['fall'],  final_width=600, final_height=600, save_dir='./composite')

✅ Saved composite/20240912_101331.jpg


In [ ]:
baseFileNames['fall'][baseFileName]

In [ ]:
imagPaths

In [ ]:
def get_person_crops(video_path, model_path="yolov8l.pt", conf_thres=0.5):
    """
    Reads a video, applies YOLO person detection, and returns cropped person images.

    Args:
        video_path (str): Path to the .mp4 video
        model_path (str): Path to YOLO model (default = yolov8n.pt)
        conf_thres (float): Confidence threshold for detection

    Returns:
        List of cropped person images (numpy arrays in BGR)
    """
    # Load YOLO
    model = YOLO(model_path)

    crops = []
    cap = cv2.VideoCapture(str(video_path))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Run YOLO detection on frame
        results = model(frame, conf=conf_thres, verbose=False)

        # Extract person detections (class 0 in COCO)
        for box in results[0].boxes:
            cls = int(box.cls[0])
            if cls == 0:  # person
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                crop = frame[y1:y2, x1:x2]
                if crop.size > 0:
                    crops.append(crop)

    cap.release()
    return crops

def resize_images(imgs, size=(300,300)):
  lst = []
  for img in imgs:
    lst.append(cv2.resize(img, size))
  return lst


def sub_sample(lst, elens=25):
    """
    Subsample a list to have at most `elens` elements, evenly spaced.
    Indices are generated first to avoid repetition.

    Args:
        lst (list): Input list
        elens (int): Desired number of elements

    Returns:
        list: Subsampled list
    """
    n = len(lst)
    if n <= elens:
        return lst, list(range(n))

    step = n / elens
    indices = [int(i * step) for i in range(elens)]
    indices = list(dict.fromkeys(indices))  # remove any accidental repetitions

    return [lst[i] for i in indices], indices  

In [ ]:
path = videos['fall'][0]
print(path)
crops = get_person_crops(path, model_path='yolo11l.pt')

In [ ]:
crops_resized = resize_images(crops, (100, 100))
crops_resized_sample, _ = sub_sample(crops_resized, 20)
array = np.concatenate(crops_resized_sample, axis=1)  # axis=1 → horizontal concat

In [ ]:
plt.figure(figsize=(40, 2))  # width=12in, height=6in
plt.imshow(cv2.cvtColor(array, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
refPaths = {'fall' : '../Kaggle/Fall/Raw_Image', 'no_fall' : '../Kaggle/No_Fall/Raw_Image'}
key = 'no_fall'
start_from = 0
crops_number = 15
crops_size = (100,100)

for en, path in enumerate(videos[key]):
    print(en, path)

    if en < start_from:
        continue

    crops = get_person_crops(path, model_path='yolo11l.pt')

    print(f'crops length {len(crops)}')
    if not crops:
        continue
    
    crops_resized = resize_images(crops, crops_size)
    crops_resized_sample, _ = sub_sample(crops_resized, crops_number)
    array = np.concatenate(crops_resized_sample, axis=1)

    prefix = path.stem
    images = crops_resized_sample + [array]
    refPath = refPaths[key]
    save_images(images, refPath=refPath, prefix=prefix)